In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 4.3 Spacetime, Minkowski Diagrams, and Four-Vectors

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume IV — Special Relativity",
    number="4.3",
    title="Spacetime, Minkowski Diagrams, and Four-Vectors",
    blurb="The geometric view: events live in spacetime, a Lorentz boost is a "
    "hyperbolic rotation that leaves the interval fixed, the light cone "
    "sorts cause from coincidence, and the quantities of physics organize "
    "themselves into four-vectors.",
    difficulty="intermediate",
    estimate="130–160 min",
)

## Notebook overview

[§4.2](lorentz-transformation.ipynb) derived the Lorentz transformation and its consequences as *algebra*. This
notebook reveals the geometry hiding underneath. Minkowski's insight, a few years after
Einstein, was that space and time are not separate stages but a single four-dimensional
**spacetime**, and that the Lorentz transformation is simply a *rotation* of that
spacetime, of an unusual hyperbolic kind. Once we see it this way, time dilation, length
contraction, and the relativity of simultaneity stop being separate effects to memorise:
they are projections onto tilted axes, read straight off a diagram.

We build that diagram, the **Minkowski diagram**, as a reusable drawing primitive, and
use it throughout: worldlines as paths through spacetime, the **light cone** as the
absolute boundary between cause and coincidence, the boosted axes closing symmetrically
toward the light line, and the **invariant hyperbolae** along which a boost slides
events. One animation sets the whole picture in motion, sweeping the boost speed and
watching the geometry rotate.

The notebook also builds the **computational toolkit** for the rest of the volume. A
four-vector is just a `numpy` array of four numbers that transform together, and its
invariant length is a contraction with the **Minkowski metric** $\eta=\operatorname{diag}
(-1,1,1,1)$. We develop the Einstein summation convention made literal by `np.einsum`,
explaining the index-string grammar carefully and verifying it against an explicit
nested sum, because the reader must be able to *trust* an index string before relying on
it. These tools, the metric, `np.einsum`, the $4\times4$ boost, recur in [§4.7](relativistic-lagrangian-fields.ipynb) (relativistic
dynamics), [§4.8](gr-capstone.ipynb) (the curved-spacetime capstone), and were already used in [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb).

Everything is in **SI units**, with $c=1/\sqrt{\mu_0\varepsilon_0}=2.998\times10^8\,$m/s;
spacetime components $(ct,x,y,z)$ are all lengths in metres, so the axes share units. One
effect here is genuinely dynamic, a boost rotating the whole diagram, and is animated;
the rest are clean stills.

> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the worldline slopes; $\cosh\varphi=\gamma$ and $\sinh\varphi=\gamma
> \beta$; the interval fixed under a boost; the norm agreeing across `np.einsum`, `@`, and
> an explicit loop; the causal classification frame-independent; the four-vector norm
> boost-invariant; the inertial worldline having the longest proper time. A ✓ is strong
> evidence; a ✗ is a prompt to *locate the discrepancy*, not a verdict.
>
> **Scope.** The geometry of spacetime and the four-vector toolkit; the paradoxes follow
> in [§4.4](paradoxes.ipynb) and relativistic dynamics in [§4.5](four-momentum-energy.ipynb)–[§4.7](relativistic-lagrangian-fields.ipynb). See Minkowski's 1908 "Space and Time";
> Taylor & Wheeler, *Spacetime Physics* {cite}`taylor_wheeler`; Nolting, *Theoretical
> Physics 4* {cite}`nolting4`; and [§4.2](lorentz-transformation.ipynb) (rapidity) and [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) (four-vectors and the
> field tensor as used).

## Theory in brief

### Spacetime and the interval as geometry

An **event** is a point in spacetime, labelled $(ct,x,y,z)$. The invariant interval of
[§4.2](lorentz-transformation.ipynb) plays the role of a squared distance, but with a crucial sign,

```{math}
:label: eq-spacetime
s^2=-(ct)^2+x^2+y^2+z^2=\eta_{\mu\nu}\,x^\mu x^\nu, \qquad \eta=\operatorname{diag}(-1,1,1,1).
```

The **Minkowski metric** $\eta$ carries the minus sign that makes time different from
space. Unlike a Euclidean distance, $s^2$ can be negative, zero, or positive, and that
sign is physics.

### The boost as a hyperbolic rotation

In the rapidity $\varphi=\operatorname{arctanh}(v/c)$ of [§4.2](lorentz-transformation.ipynb), the boost on $(ct,x)$ is

```{math}
:label: eq-hyperbolic-rotation
\Lambda(\varphi)=\begin{pmatrix}\cosh\varphi & -\sinh\varphi\\ -\sinh\varphi & \cosh\varphi\end{pmatrix}, \qquad \cosh\varphi=\gamma, \quad \sinh\varphi=\gamma\beta.
```

This is to the interval exactly what an ordinary rotation $\begin{psmallmatrix}\cos & -\sin
\\ \sin & \cos\end{psmallmatrix}$ is to Euclidean distance: a **hyperbolic rotation** that
preserves $-(ct)^2+x^2$ and slides events along hyperbolae instead of circles.

### The light cone and causal structure

The null surface $ct=\pm|x|$ is the **light cone**. The sign of $s^2$ classifies any pair
of events absolutely,

```{math}
:label: eq-light-cone
s^2<0 \ \text{timelike (causally connectable)}, \quad s^2=0 \ \text{null (light)}, \quad s^2>0 \ \text{spacelike (no causal link)}.
```

Because $s^2$ is invariant, every observer agrees on this classification: the causal
structure of spacetime is frame-independent.

### Minkowski diagrams

Plot $ct$ vertically against $x$ horizontally. A stationary object is a vertical
worldline, a moving one is tilted, light is at $45^\circ$. A boost to $v=\beta c$ tilts
the primed axes **symmetrically** toward the light line,

```{math}
:label: eq-minkowski-diagram
ct'\text{-axis: } ct=x/\beta \ (\text{slope } 1/\beta), \qquad x'\text{-axis: } ct=\beta x \ (\text{slope } \beta),
```

with slope product $1$. Time dilation and length contraction are projections onto these
tilted axes, and the $x'$-axis is precisely the set of events $S'$ calls simultaneous, so
the relativity of simultaneity becomes visible.

### Four-vectors

The position four-vector $x^\mu=(ct,x,y,z)$ transforms by the Lorentz matrix, and its
norm $\eta_{\mu\nu}x^\mu x^\nu$ is the invariant interval,

```{math}
:label: eq-st-four-vectors
x'^\mu=\Lambda^\mu{}_\nu\,x^\nu, \qquad \eta_{\mu\nu}x'^\mu x'^\nu=\eta_{\mu\nu}x^\mu x^\nu.
```

*Any* four quantities that transform this way form a four-vector (four-velocity and
four-momentum follow in [§4.5](four-momentum-energy.ipynb)). The payoff: write a law as a four-vector equation and it
holds, unchanged, in every inertial frame.

### Computational tools for four-vectors

We make this literal. A four-vector is a `numpy` 4-array; the metric $\eta$ is an explicit
array; the Einstein summation convention (repeated index summed, free index kept) is
`np.einsum` with an **index string**,

```{math}
:label: eq-einsum-tools
\eta_{\mu\nu}x^\mu x^\nu \ \to\ \texttt{np.einsum('a,ab,b->', x, eta, x)}, \qquad x_\mu=\eta_{\mu\nu}x^\nu \ \to\ \texttt{np.einsum('ab,b->a', eta, x)} .
```

A repeated letter is summed, the letters after `->` are kept, so an empty right side is a
scalar. Use `@` for plain matrix-vector products ($\Lambda x$) and `np.einsum` for the
general index gymnastics. Exercise 4 verifies the index string against an explicit nested
loop, because trusting the notation is the whole point.

## Setup

Data and instruments only: the series palette, the Minkowski metric
$\eta=\operatorname{diag}(-1,1,1,1)$ that fixes this notebook's signature, and the
Lorentz factor $\gamma(\beta)$ of [§4.2](lorentz-transformation.ipynb), which enters
here as the *known* quantity the geometry is checked against. The notebook's own
toolkit is not here: you write the $2\times2$ hyperbolic rotation `boost2` in
Exercise 2, the Minkowski norm `four_norm` in Exercise 4, and the $4\times4$ boost
`boost4` in Exercise 5. Every change of frame, every interval, and every causal
classification from there on runs on those three. The only randomness in the notebook
is the arbitrary four-vector of Exercise 9, seeded there.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import draw, validate
from ecp.animate import show

# data: the series palette
ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT

# data: the Minkowski metric, the signature convention (−, +, +, +) this notebook fixes
ETA = np.diag([-1.0, 1.0, 1.0, 1.0])


# instrument: the Lorentz factor carried over from §4.2, one line of arithmetic
# transcribed from its definition. It is not built here and is nobody's lesson here:
# it is the known number against which the hyperbolic rotation of Exercise 2 (cosh φ,
# sinh φ) and the time dilation of Exercise 7 are checked.
def gamma(beta):
    """The Lorentz factor γ = 1/sqrt(1 − β^2) as a function of β = v/c (eq-hyperbolic-rotation).

    Parameters
    ----------
    beta : float or numpy.ndarray
        Speed as a fraction of c (|β| < 1).

    Returns
    -------
    float or numpy.ndarray
        The Lorentz factor, matching the shape of ``beta``.
    """
    return 1.0 / np.sqrt(1.0 - beta**2)

## Exercise 1 — The Minkowski diagram and worldlines (worked)

A Minkowski diagram plots $ct$ upward against $x$ rightward, so that the path of an
object through spacetime, its **worldline**, is a curve on the page. A stationary object
stays at fixed $x$, a vertical line; an object at $v=\beta c$ traces $x=\beta\,ct$, a line
of slope $1/\beta$ in $ct$-vs-$x$, steeper than $45^\circ$ because $\beta<1$; and light
moves at $45^\circ$, the **light cone** ({numref}`fig-st-worldlines`). We build the
reusable `ecp.draw` spacetime-diagram primitive here and use it for the rest of the
volume.

**Part a)** Draw the diagram on the ruled grid with `draw.spacetime_diagram`, and add
three worldlines with `draw.worldline`: a stationary object, one moving at $v=0.5c$,
and a light ray.

**Part b)** Confirm the slopes numerically, as plain `numpy` ratios: the light
worldline at $45^\circ$ (slope $1$ in $ct$-vs-$x$) and the massive one at slope
$1/\beta=2$, steeper, so that it stays inside the cone.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    slope_light, 1.0, "the light worldline sits at 45° (slope 1 in ct vs x)", rtol=1e-12
)
validate.check(
    slope_move > slope_light,
    "a massive worldline is steeper than the light line — it stays inside the cone",
)

## Exercise 2 — The boost as a hyperbolic rotation (worked)

Here is the central geometric fact. In the rapidity $\varphi=\operatorname{arctanh}
(\beta)$ of [§4.2](lorentz-transformation.ipynb), the Lorentz boost {eq}`eq-hyperbolic-rotation` is built from $\cosh
\varphi$ and $\sinh\varphi$ exactly as an ordinary rotation is built from $\cos$ and
$\sin$, with $\cosh\varphi=\gamma$ and $\sinh\varphi=\gamma\beta$. A boost is therefore a
**hyperbolic rotation** of the spacetime plane: as the speed grows, the primed axes close
symmetrically toward the $45^\circ$ light line, and a fixed event's coordinates slide
along an invariant hyperbola ({numref}`fig-st-boost`).

The matrix built below is checked against the $\gamma$-form of [§4.2](lorentz-transformation.ipynb),
$\Lambda=\begin{psmallmatrix}\gamma & -\gamma\beta\\ -\gamma\beta &
\gamma\end{psmallmatrix}$, which is the same boost written in the variable $\beta$
instead of $\varphi$; the two agreeing entry by entry is exactly the statement
$\cosh\varphi=\gamma$, $\sinh\varphi=\gamma\beta$. The worked speed is $\beta=0.6$.

**Part a)** Write `boost2(beta)`, returning the $2\times2$ boost $\Lambda(\varphi)$ of
{eq}`eq-hyperbolic-rotation` acting on the column $(ct,x)$: form the rapidity
$\varphi=\operatorname{arctanh}\beta$ with `np.arctanh`, then assemble the explicit
array $\begin{psmallmatrix}\cosh\varphi & -\sinh\varphi\\ -\sinh\varphi &
\cosh\varphi\end{psmallmatrix}$ with `np.cosh` and `np.sinh`. **Write this one
yourself** — the implementation is the lesson, and every change of frame in this
notebook goes through it.

**Part b)** Confirm $\cosh\varphi=\gamma$ and $\sinh\varphi=\gamma\beta$ at
$\beta=0.6$, and check with `np.allclose` that your matrix equals the $\gamma$-form
boost entry for entry.

**Part c)** Set the picture in motion: sweep $\beta$ from $0$ to $0.9$, closing the
primed axes toward the light line while a fixed event rides its invariant hyperbola.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    np.array([np.cosh(phi), np.sinh(phi)]),
    np.array([gamma(beta), gamma(beta) * beta]),
    "the boost is a hyperbolic rotation: cosh φ = γ, sinh φ = γβ",
    rtol=1e-9,
)
validate.check(
    np.allclose(hyp, std), "the rapidity-form and γ-form boosts are the same matrix"
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Invariant hyperbolae and the geometry of dilation and contraction (worked)

Where an ordinary rotation moves points along circles of fixed radius, a boost moves
events along the **invariant hyperbolae** $-(ct)^2+x^2=\text{const}$ {eq}`eq-spacetime`.
These curves are the level sets of the interval, and a boost cannot leave one. Reading
time dilation and length contraction off the diagram is now a matter of projecting onto
the tilted axes: a unit tick of the moving clock reaches *higher* up our $ct$-axis than
its own, and a moving rod projects *shorter* onto our $x$-axis ({numref}`fig-st-hyperbola`).

**Part a)** Take the event $(ct,x)=(1.2,0.4)\,$m, boost it to $0.5c$ with the `boost2`
you wrote in Exercise 2, and confirm with the explicit combination $-(ct)^2+x^2$ that
its interval is unchanged: the boost has slid it along one hyperbola and cannot take it
off.

**Part b)** Draw the timelike and spacelike hyperbolae $-(ct)^2+x^2=\mp1$ together with
the boosted axes and the event's two positions, so the algebra of Part a) becomes a
picture.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    s2_after,
    s2_before,
    "boosts slide events along invariant hyperbolae (s² fixed)",
    rtol=1e-9,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The four-vector toolkit: `np.einsum` and the metric (worked)

Now we build the computational language of the rest of the volume, in the careful,
tools-with-a-note style of [§3.5](../03-electrodynamics/multipole-expansion.ipynb). A four-vector is a `numpy` array of four numbers; the
metric $\eta=\operatorname{diag}(-1,1,1,1)$ is an explicit array; and the inner product
$\eta_{\mu\nu}x^\mu x^\nu$ is a sum over repeated indices, which `np.einsum` writes
literally. The **index string** is the grammar: each letter is an index, a letter
repeated on the input side is summed over, and the letters after `->` are kept, so
`'a,ab,b->'` (empty right side) contracts everything to a scalar
{eq}`eq-einsum-tools`. Raising or lowering an index, $x_\mu=\eta_{\mu\nu}x^\nu$, is
`'ab,b->a'`, which keeps one free index and flips the sign of the time component.

The four-vector used below is $x^\mu=(1.3,0.5,0.2,0)\,$m, timelike, so its norm should
come out negative.

**Part a)** Write `four_norm(x)`, returning the Minkowski norm $\eta_{\mu\nu}x^\mu
x^\nu$ {eq}`eq-einsum-tools` of a length-4 array as the single contraction
`np.einsum('a,ab,b->', x, ETA, x)` — the summation convention made literal, negative
for timelike, zero for null, positive for spacelike four-vectors. It is the workhorse
of every exercise that follows.

**Part b)** Earn the right to trust that index string: compute the same norm two more
ways, as the chained matrix product `x @ ETA @ x` and as an explicit double
`for`-loop over $\mu$ and $\nu$, and confirm with `np.allclose` that all three agree.
An index string is worth using only once it has matched the loop it abbreviates.

**Part c)** Lower the index with `np.einsum('ab,b->a', ETA, x)` and watch the time
component change sign, the metric's minus sign doing its one job.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    np.array([einsum_norm, matmul_norm]),
    loop_norm,
    "the four-vector norm via np.einsum, @, and an explicit sum all agree",
    rtol=1e-12,
)
validate.close(
    x_lower[0],
    -x4[0],
    "lowering the index flips the sign of the time component",
    rtol=1e-12,
)

## Exercise 5 — The causal structure of spacetime (worked)

The sign of the interval is not a convention but the deepest invariant in physics. Two
events separated by a **timelike** interval ($s^2<0$) can be connected by something
slower than light, so one can cause the other; a **spacelike** separation ($s^2>0$) lies
outside the light cone, so no signal can pass and neither event can influence the other;
a **null** separation ($s^2=0$) is exactly a light ray {eq}`eq-light-cone`. Because $s^2$
is invariant, this classification is the *same in every frame*: the causal order of
spacetime is absolute even though time and space separately are not
({numref}`fig-st-causal`).

Testing that claim needs a boost that acts on all four components, not just the
$(ct,x)$ pair of Exercise 2. The $2\times2$ hyperbolic rotation extends to one in the
obvious way: a boost along $x$ does nothing at all to $y$ and $z$, so it sits as a
block in the corner of a $4\times4$ matrix whose transverse part is the identity.

**Part a)** Write `boost4(beta)`, that $4\times4$ Lorentz boost along $x$
{eq}`eq-st-four-vectors`: $\gamma$ in the $(0,0)$ and $(1,1)$ slots, $-\gamma\beta$ in
the $(0,1)$ and $(1,0)$ slots — the Exercise 2 block, now in $\gamma$-form using the
Setup's `gamma` — and $1$ on the remaining diagonal, $0$ everywhere else. **Write this
one yourself** — the implementation is the lesson, and every four-vector in the rest of
the notebook changes frame through it.

**Part b)** Take three events, one per causal class: $(2,0.5,0,0)$ inside the cone,
$(0.5,2,0,0)$ outside it, and $(1.5,1.5,0,0)$ exactly on it. Classify each separation
from the origin by the sign of $s^2$, computed as the `four_norm` you wrote in
Exercise 4 applied to the difference four-vector, then boost both endpoints to a frame
at $0.7c$ and confirm the sign — and so the causal class — is unchanged.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    signs_ok,
    "the timelike/spacelike/null classification is frame-independent (causal structure is absolute)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — The four-vector norm is boost-invariant (student)

The whole reason four-vectors matter is that their norm is a Lorentz scalar: apply any
boost and $\eta_{\mu\nu}x^\mu x^\nu$ comes out the same {eq}`eq-st-four-vectors`. This is
why writing a physical law as a four-vector equation makes it automatically frame-
independent, the strategy that drives all of relativistic mechanics and the covariant
Maxwell equations of [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb).

**Part a)** Apply the `boost4` you wrote in Exercise 5, at $\beta=0.8$, to the
four-vector $x^\mu=(2,1,0.5,0.3)\,$m using the matrix product `@`, and confirm with
your Exercise 4 `four_norm` that the norm before and after the boost agree — even
though every one of the first two components has changed.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    norm_after,
    norm_before,
    "the four-vector norm is invariant under a Lorentz boost",
    rtol=1e-10,
)

## Exercise 7 — Reading a paradox off the diagram (student)

The Minkowski diagram dissolves the twin paradox before we even compute. One twin stays
home (a straight, vertical worldline); the other flies out at $v=\beta c$ and returns (a
bent worldline). The proper time each twin ages is the "length" of their worldline in the
spacetime metric, $c\,\tau=\int\sqrt{(c\,dt)^2-dx^2}$, and because of the minus sign this
**bent** path is *shorter* than the straight one, the opposite of the Euclidean rule that
a straight line is the shortest distance ({numref}`fig-st-twins`). The travelling twin
returns younger. We meet the full resolution in [§4.4](paradoxes.ipynb); here we read it off the geometry.

The two proper times are known in closed form and need no integration: the stay-home
twin is inertial, so their proper time is just the coordinate time $T$, while each of
the traveller's two legs runs at constant speed and is dilated by the same $\gamma$, so
the traveller ages $T/\gamma$. The round trip below has $T=2$ (in units of $ct$) at
$\beta=0.6$.

**Part a)** Draw the two worldlines with the spacetime-diagram primitive: the stay-home
twin's straight vertical line, and the traveller's outbound and inbound legs meeting at
the turnaround.

**Part b)** Put numbers on the picture. Compute both proper times and confirm, as an
explicit `numpy` comparison, that the bent worldline is the one that ages less.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    tau_travel < tau_home,
    "the inertial (straight) worldline has the longest proper time — the traveller ages less",
)

## Exercise 8 — Velocity as a slope; the speed limit, geometrically (student)

The light cone is the cosmic speed limit drawn as geometry. A massive particle's
worldline must be **timelike** everywhere, steeper than $45^\circ$ in $ct$-vs-$x$, which
is the same as saying its speed $\beta<1$; light rides exactly on the cone at $45^\circ$;
and nothing has a spacelike worldline, because that would mean $\beta>1$ and an interval
that some frame sees running backward in time. The speed limit is not a law imposed on
top of spacetime, it *is* the shape of spacetime.

A worldline's causal character is read off any step along it, and the convenient step
for speed $\beta$ is the four-vector $(1,\beta,0,0)$: advance $ct$ by one unit and $x$
by $\beta$.

**Part a)** For the candidate speeds $\beta=0.2,\,0.5,\,0.9,\,1$, classify each
worldline by the sign of the interval of that step, computed with your Exercise 4
`four_norm`, and confirm with a `numpy` comparison that every massive worldline
($\beta<1$) comes out timelike — inside the cone — while $\beta=1$ comes out null, on
it.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    all_massive_timelike,
    "massive particles travel slower than light — their worldlines lie inside the cone",
)

## Exercise 9 — Geometry is the deeper language

Step back and see what the geometry bought us. Everything in [§4.2](lorentz-transformation.ipynb), the transformation,
time dilation, length contraction, velocity addition, was the *shadow* of a single
structure: spacetime carries an indefinite metric $\eta=\operatorname{diag}(-1,1,1,1)$,
and Lorentz boosts are its rotations, hyperbolic rather than circular. The light cone is
the absolute causal framework, the invariant hyperbolae are the orbits of a boost, and
four-vectors are the natural objects: any law written as a four-vector equation is
relativistically invariant by construction. This is the language in which [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) wrote the
field tensor and in which all of relativistic dynamics is cast.

One fact underlies all the rest, and it deserves the last word. Everything so far used
four-vectors we chose; the invariance holds for four-vectors nobody chose.

**Part a)** Draw an arbitrary four-vector from a seeded generator and confirm in a
single check that your Exercise 5 `boost4`, at $\beta=0.55$, leaves its `four_norm`
unchanged to machine precision — the invariance that makes geometry, not algebra, the
true home of relativity.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.check(
    preserved,
    "a boost preserves the Minkowski norm — four-vector equations are automatically invariant",
)

## Notebook summary

- **The Minkowski diagram** {eq}`eq-minkowski-diagram`: $ct$ versus $x$, worldlines as
  paths; light at $45^\circ$, massive objects steeper (slope $1/\beta>1$), built as the
  reusable `draw.spacetime_diagram` primitive (with `worldline`, `boosted_axes`,
  `hyperbola`), reused in [§4.4](paradoxes.ipynb) and the GR capstone.
- **The boost as a hyperbolic rotation** {eq}`eq-hyperbolic-rotation`: $\Lambda$ is
  $\cosh\varphi,\sinh\varphi$ with $\cosh\varphi=\gamma$, $\sinh\varphi=\gamma\beta$; the
  primed axes close symmetrically toward the light line and events slide along the
  **invariant hyperbolae** $-(ct)^2+x^2=$ const, the interval fixed under the boost.
- **The four-vector toolkit** {eq}`eq-einsum-tools`: four-vectors as `numpy` 4-arrays,
  the metric $\eta=\operatorname{diag}(-1,1,1,1)$ explicit, and the norm $\eta_{\mu\nu}
  x^\mu x^\nu$ as `np.einsum('a,ab,b->', x, ETA, x)`, verified equal to `x @ ETA @ x` and
  an explicit loop ($-1.40\,$m²); lowering an index `'ab,b->a'` flips the time sign.
- **Causal structure** {eq}`eq-light-cone`: the sign of $s^2$ sorts every pair into
  timelike / null / spacelike, and a $0.7c$ boost leaves the class unchanged, the
  absolute causal order of spacetime.
- **Invariance and consequences** {eq}`eq-st-four-vectors`: the four-vector norm is boost-
  invariant ($0.8c$, to $10^{-10}$); the inertial twin's straight worldline has the
  *longest* proper time (the traveller ages by $T/\gamma$); and massive worldlines lie
  strictly inside the light cone, the speed limit as geometry.

## Outlook

- **The paradoxes, resolved by computation ([§4.4](paradoxes.ipynb)).** The twin and pole-and-barn, settled
  by proper-time integrals along the worldlines drawn here.
- **Four-velocity and four-momentum; $E=mc^2$ ([§4.5](four-momentum-energy.ipynb)).** The next four-vectors, with the
  metric and `np.einsum` toolkit of Exercise 4 carried straight over.
- **Relativistic dynamics and fields ([§4.7](relativistic-lagrangian-fields.ipynb)).** Where four-vectors meet forces, reusing the
  tensor machinery; the field tensor $F^{\mu\nu}$ of [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) lives in exactly this geometry.
- **The curved-spacetime capstone ([§4.8](gr-capstone.ipynb)).** The same metric, now position-dependent, with
  `np.einsum` contractions over a curved $\eta\to g_{\mu\nu}$.
- **The Lorentz group** as the symmetry group of spacetime, rotations and boosts together
  (a pointer).

In [ ]:
from ecp.style import footer

footer()